# 02 — Exploratory Data Analysis and SQL
**Goal:** understand class balance, launch history, and differences across launch sites and years.

In [ ]:
from pathlib import Path
import sys, sqlite3
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent
df = pd.read_csv(ROOT / "data" / "spacex_launches_raw.csv", parse_dates=["date_utc"])
df["landing_success"] = df["landing_success"].map({"True": 1, "False": 0, True: 1, False: 0})
df["launch_year"] = pd.to_numeric(df["launch_year"], errors="coerce")
known = df[df["landing_success"].notna() & (df["upcoming"] != True)].copy()
print("All launches:", len(df), "| Known landing outcomes:", len(known))
display(known.head())

In [ ]:
# Target distribution
counts = known["landing_success"].value_counts().sort_index()
ax = counts.rename(index={0:"Unsuccessful",1:"Successful"}).plot(kind="bar", rot=0)
ax.set_title("Known first-stage landing outcomes")
ax.set_ylabel("Launch count")
plt.tight_layout()
plt.show()

In [ ]:
# Success rate by launch year; show only groups with at least 3 known outcomes
by_year = (known.dropna(subset=["launch_year"])
           .groupby("launch_year")["landing_success"]
           .agg(["count","mean"]).query("count >= 3"))
by_year["success_rate_pct"] = by_year["mean"] * 100
display(by_year[["count","success_rate_pct"]].round(1))
by_year["success_rate_pct"].plot(marker="o", title="Landing success rate by year")
plt.ylabel("Success rate (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Site-level comparison, suppressing tiny groups
by_site = (known.groupby("launchpad_name")["landing_success"]
           .agg(["count","sum","mean"]).query("count >= 3")
           .sort_values("mean", ascending=False))
by_site = by_site.rename(columns={"count":"known_outcomes","sum":"successful_landings","mean":"success_rate"})
display((by_site.assign(success_rate_pct=100*by_site["success_rate"])
         [["known_outcomes","successful_landings","success_rate_pct"]]).round(1))

In [ ]:
# SQL examples with SQLite
db_path = ROOT / "data" / "spacex_capstone.sqlite"
with sqlite3.connect(db_path) as con:
    known.to_sql("launches", con, if_exists="replace", index=False)
    sql_result = pd.read_sql_query('''
        SELECT launchpad_name, COUNT(*) AS n,
               ROUND(100.0 * AVG(landing_success), 1) AS success_rate_pct
        FROM launches
        WHERE landing_success IS NOT NULL
        GROUP BY launchpad_name
        HAVING COUNT(*) >= 3
        ORDER BY success_rate_pct DESC
    ''', con)
display(sql_result)

## Interpretation prompts
- Are trends stable when restricting to launch sites with sufficient observations?
- Could launch site differences be confounded by launch year, rocket version, or mission type?
- Do not write a site as “best” based only on a small sample or unadjusted success rate.